# vLLM-V1


根据 vLLM 官方对 v1 技术介绍[vllm-seven-meetsup](https://docs.google.com/presentation/d/1e3CxQBV3JsfGp30SwyvS3eM_tW-ghOhJ9PAJGK6KR54/edit?slide=id.g31455c8bc1e_1_28#slide=id.g31455c8bc1e_1_28)

1. step0 时, 对于 R1/R2 都是满 context 填充的, R3 是 chunk-prefill
2. step1 时, 混合 Decoding(R1/R2), chunked-prefill(R3)
2. step2 时, 混合 Decoding(R1/R2), chunked-prefill(R3)
2. step3 时, 混合 Decoding(R1/R2/R3)

这里的实现过程里，Chunk-Prefill 达到了目的。 R1 在多个 step 才输出 prefill-predict-next-token, 在这个过程里，Decoding 可能输出了多个 token

> prefill request into equal sized chunks, and decode-maximal batching


<div style="text-align: center;">
    <figure>
      <img src="./vLLM-Chunked-Prefill.png" width="60%">
      <figcaption>vLLM Chunked-Prefill</figcaption>
    </figure>
</div>

## 实现分析

1. 多个请求的输入被拉长了一条序列，token budget 决定了 forward 时非 attention 的计算量
2. decoding 请求优先并入到 batch 中
3. chunked 大小不是固定的。

vLLM 虽然说是消除了 P/D 的概念，实现上仍然需要写特定的 attention kernel 

代码在目录 `vLLM-V1` 文件夹更新